# Simulation

A stochastic market and a valid book become arrays of paths. The market is
`examples.sample_market(stochastic=True)`. That forward rolls with
`ForwardRoll.FORWARD_VOLATILITY`, which is what a path needs. The book is the
sample collar, outright, and Asian. The first simulation date is the anchor.

`simulate` returns a `Simulation`. `spot` and `mtm` are shaped `[path, step]`.
An unmargined book has `margin is None`.


In [ ]:
import logging
import math
import tempfile
from datetime import timedelta
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import matplotlib.pyplot as plt
import numpy as np
import sjd
from sjd import examples, stats

sjd.set_log_level("INFO")

market = examples.sample_market(stochastic=True)
environment = market["environment"]
dates = market["simulation_dates"]
anchor = market["anchor"]
portfolio = sjd.Portfolio(examples.sample_contracts(market["forward"], market)["contracts"])
sjd.require_valid(environment)
sjd.require_valid(portfolio, environment)

assert dates[0] == anchor
assert dates == sorted(dates)
print(repr(portfolio))
print("dates", len(dates))


## One run

`paths` is the number of draws. Antithetic sampling is on, so an odd count is
rounded up to an even number of paths. Every path starts at today's forward
and today's portfolio value. `exposure_steps` and `margin_reset_steps` are the
same index on a run with no holding period.


In [ ]:
simulation = sjd.simulate(portfolio, environment, dates, paths=16, seed=3)
today = portfolio.get_value(environment)
spot_today = market["forward"].get_forward(anchor)

assert simulation.paths == 16
assert simulation.steps == len(dates)
assert simulation.dates == dates
assert simulation.antithetic
assert simulation.seed == 3
assert simulation.margin is None
assert simulation.by_trade is None
assert simulation.spot.shape == (simulation.paths, simulation.steps)
assert simulation.mtm.shape == simulation.spot.shape
assert np.allclose(simulation.spot[:, 0], spot_today)
assert np.allclose(simulation.mtm[:, 0], today)
assert np.array_equal(simulation.exposure_steps, np.arange(len(dates)))
assert np.array_equal(simulation.margin_reset_steps, simulation.exposure_steps)
print(repr(simulation))
print(f"value today {today:.2f}")


## Spot and mark-to-market

Each line is one path. The paths meet on the anchor date and then spread out.


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True)
for path in range(simulation.paths):
    axes[0].plot(simulation.dates, simulation.spot[path], color="C0", alpha=0.4)
    axes[1].plot(simulation.dates, simulation.mtm[path], color="C1", alpha=0.4)
axes[0].set_title("Spot")
axes[1].set_title("Mark to market")
axes[0].set_ylabel("Forward")
axes[1].set_ylabel("Value")
figure.autofmt_xdate()
figure.tight_layout()
figure


## By trade

`by_trade=True` adds a third axis, one slice per contract. The slices sum to `mtm`.


In [ ]:
detailed = sjd.simulate(
    portfolio, environment, dates, paths=6, seed=2, antithetic=False, by_trade=True
)
assert detailed.trades == 4
assert detailed.by_trade.shape == (6, len(dates), 4)
assert np.allclose(detailed.by_trade.sum(axis=2), detailed.mtm, atol=1e-6)
print("by trade", detailed.by_trade.shape)


## Antithetic paths and the seed

Path `i` and path `i + paths // 2` are drawn from opposite normals. They share
today's forward. After that, the sum of their log spots is the same for every
pair: each pair sits the same distance either side of the forward curve.

The same seed repeats the arrays. A different seed does not. An omitted seed
is drawn, stored on `simulation.seed`, and passing it back repeats the run.
An odd requested count is stored as the next even count.


In [ ]:
half = simulation.paths // 2
spot = np.asarray(simulation.spot)
pair_log = np.log(spot[:half]) + np.log(spot[half:])
assert np.allclose(pair_log, pair_log[0])
assert not np.allclose(spot[0, 1:], spot[half, 1:])

same = sjd.simulate(portfolio, environment, dates, paths=16, seed=3)
other = sjd.simulate(portfolio, environment, dates, paths=16, seed=9)
assert np.array_equal(same.spot, simulation.spot)
assert np.array_equal(same.mtm, simulation.mtm)
assert not np.array_equal(other.mtm[:, 1:], simulation.mtm[:, 1:])

drawn = sjd.simulate(portfolio, environment, dates, paths=8, antithetic=False)
assert isinstance(drawn.seed, int)
repeated = sjd.simulate(
    portfolio, environment, dates, paths=8, seed=drawn.seed, antithetic=False
)
assert repeated.seed == drawn.seed
assert np.array_equal(repeated.mtm, drawn.mtm)

rounded = sjd.simulate(portfolio, environment, dates, paths=7, seed=1)
assert rounded.paths == 8
assert rounded.antithetic
print(f"pair log sum {pair_log[0, -1]:.6f}")
print(f"drawn seed {drawn.seed}")


## Threads and sampling

One thread and four threads write the same arrays. `sampling="sobol"` and
`sampling="pseudo"` use the same path count and seed and produce different
paths. The error below is the distance of the mean terminal spot from the
forward on that date. At 64 paths the Sobol mean is closer.


In [ ]:
one = sjd.simulate(portfolio, environment, dates, paths=16, seed=4, threads=1)
many = sjd.simulate(portfolio, environment, dates, paths=16, seed=4, threads=4)
assert np.array_equal(one.spot, many.spot)
assert np.array_equal(one.mtm, many.mtm)

pseudo = sjd.simulate(
    portfolio, environment, dates, paths=64, seed=4, antithetic=False, threads=1, sampling="pseudo"
)
sobol = sjd.simulate(
    portfolio, environment, dates, paths=64, seed=4, antithetic=False, threads=1, sampling="sobol"
)
forward = market["forward"].get_forward(dates[-1])
pseudo_error = abs(float(pseudo.spot[:, -1].mean()) - forward)
sobol_error = abs(float(sobol.spot[:, -1].mean()) - forward)
assert np.isfinite(pseudo.mtm).all()
assert np.isfinite(sobol.mtm).all()
assert not np.array_equal(pseudo.mtm, sobol.mtm)
assert sobol_error < pseudo_error
print(f"pseudo error {pseudo_error:.4f}")
print(f"sobol error  {sobol_error:.4f}")


## Profiles and a narrower array

`aggregates=True` keeps one number per step: the mean spot, the mean
mark-to-market, expected exposure, and potential future exposure. The path
cube is not returned. `float32=True` stores the same run in half the bytes.


In [ ]:
full = sjd.simulate(
    portfolio, environment, dates, paths=20, seed=6, antithetic=False, threads=1
)
profiles = sjd.simulate(
    portfolio, environment, dates, paths=20, seed=6, antithetic=False, threads=1, aggregates=True
)
assert profiles.aggregated
assert profiles.spot.shape == (full.steps,)
assert profiles.mtm.shape == (full.steps,)
assert profiles.ee.shape == (full.steps,)
assert profiles.pfe.shape == (full.steps,)
assert np.allclose(profiles.spot, full.spot.mean(axis=0))
assert np.allclose(profiles.mtm, full.mtm.mean(axis=0))
assert np.allclose(profiles.ee, stats.expected_exposure(full.mtm))
assert np.allclose(profiles.pfe, stats.potential_future_exposure(full.mtm))

narrow = sjd.simulate(
    portfolio, environment, dates, paths=20, seed=6, antithetic=False, threads=1, float32=True
)
assert narrow.mtm.dtype == np.float32
assert narrow.spot.dtype == np.float32
assert narrow.mtm.nbytes * 2 == full.mtm.nbytes
assert np.allclose(narrow.mtm, full.mtm, rtol=1e-6, atol=1e-4)
print(f"float64 {full.mtm.nbytes} bytes")
print(f"float32 {narrow.mtm.nbytes} bytes")


## Margin and a holding period

A margined portfolio returns a `margin` array of the same shape as `mtm`.
`holding_period` is a number of business days. The calendar inserts a margin
reset date that many business days before each later simulation date.
`exposure_steps` still points at the requested dates. `margin_reset_steps`
points at the reset dates. A holding period with no calendar raises
`ValueError`.


In [ ]:
calendar = market["calendar"]
margin = sjd.SimpleMargin(anchor, True, 50_000.0, True, 50_000.0, 10_000.0, 1_000.0)
margined = sjd.Portfolio(portfolio.contracts(), margin)
sjd.require_valid(margined, environment)
held = sjd.simulate(
    margined,
    environment,
    dates,
    paths=8,
    seed=13,
    holding_period=10,
    holding_period_calendar=calendar,
)
reset = calendar.advance(dates[1], -10)
assert held.margin.shape == held.mtm.shape
assert stats.exposure_dates(held) == dates
assert held.dates[int(held.margin_reset_steps[1])] == reset
assert not np.array_equal(held.margin_reset_steps, held.exposure_steps)
assert np.allclose(stats.exposure(held), held.mtm - held.margin)
print(f"reset {reset}")
print("simulation dates", len(held.dates))


## A path with no new random numbers

`PortfolioEvolver.value_spot_path` marks the book to a forward the caller
supplies, one spot per simulation date. The curve's own forwards recover
today's value at the anchor. `value_spot_paths` does the same for every row
of an array. Marking the simulated spots again recovers `mtm`.


In [ ]:
generator = sjd.SingleFactorEconomicScenarioGenerator(environment)
evolver = sjd.PortfolioEvolver(portfolio, generator, dates)
sjd.require_valid(evolver)
curve_spots = [market["forward"].get_forward(day) for day in dates]
along_curve = evolver.value_spot_path(curve_spots)
assert np.allclose(along_curve.spot[0], curve_spots)
assert math.isclose(float(along_curve.mtm[0, 0]), portfolio.get_value(environment))

supplied = evolver.value_spot_paths(np.asarray(simulation.spot))
assert supplied.paths == simulation.paths
assert np.allclose(supplied.mtm, simulation.mtm, rtol=1e-8, atol=1e-4)
print(f"along the curve {float(along_curve.mtm[0, 0]):.2f}")


## What can go wrong

Three inputs are refused with `ValidationError` and a report. A deterministic
forward reports `PATH_GENERATOR_FORWARD_NOT_STOCHASTIC`. A first date that is
not the anchor reports `SIMULATION_ANCHOR_MISMATCH`. A last date past the
curve reports `PATH_GENERATOR_DATES_AFTER_END`. A holding period with no
calendar is an argument error: `ValueError`, and the object has no report for it.


In [ ]:
deterministic = examples.sample_market()
deterministic_book = sjd.Portfolio(
    examples.sample_contracts(deterministic["forward"], deterministic)["contracts"]
)
sjd.require_valid(deterministic_book, deterministic["environment"])

def refused(book, rates, when, code):
    try:
        sjd.simulate(book, rates, when, paths=4, seed=1)
    except sjd.ValidationError as error:
        assert any(issue.code == code for issue in error.report.issues)
        print(code)
        return error.report
    raise AssertionError(code)

refused(
    deterministic_book,
    deterministic["environment"],
    deterministic["simulation_dates"],
    "PATH_GENERATOR_FORWARD_NOT_STOCHASTIC",
)
refused(
    portfolio,
    environment,
    [anchor + timedelta(days=1), dates[-1]],
    "SIMULATION_ANCHOR_MISMATCH",
)
late_report = refused(
    portfolio,
    environment,
    [anchor, dates[-1] + timedelta(days=800)],
    "PATH_GENERATOR_DATES_AFTER_END",
)
try:
    sjd.simulate(margined, environment, dates, paths=4, seed=1, holding_period=10)
except ValueError as error:
    assert not isinstance(error, sjd.ValidationError)
    assert "holding_period_calendar" in str(error)
    print(type(error).__name__)
else:
    raise AssertionError("a holding period needs a calendar")
late_report


## What it logs

`simulation start` records the requested paths, the date count, whether
antithetic sampling is on, and the seed. `simulation end` records the path
count that was stored and the setup and path timings. Both are `INFO` on
`sjd._core`, and both are delivered when `simulate` returns.


In [ ]:
class Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []

    def emit(self, record):
        self.records.append(record)

collected = Collect()
logging.getLogger("sjd._core").addHandler(collected)
logged = sjd.simulate(portfolio, environment, dates, paths=4, seed=1, antithetic=False)
messages = [record.getMessage() for record in collected.records]
logging.getLogger("sjd._core").removeHandler(collected)

assert any(
    message.startswith("simulation start") and "paths=4" in message and "seed=1" in message
    for message in messages
)
assert any(
    message.startswith("simulation end") and "setup_ms=" in message and "paths_ms=" in message
    for message in messages
)
for message in messages:
    print(message)
print(f"metadata seed {logged.metadata['inputs']['seed']}")


## Save and reload

The arrays go to a `.npz` file in a temporary directory. Notebook 17 stores a
whole run, including the metadata, and reads it back.


In [ ]:
scratch = tempfile.TemporaryDirectory()
path = Path(scratch.name) / "paths.npz"
np.savez(path, spot=np.asarray(simulation.spot), mtm=np.asarray(simulation.mtm))
with np.load(path) as loaded:
    assert np.array_equal(loaded["spot"], np.asarray(simulation.spot))
    assert np.array_equal(loaded["mtm"], np.asarray(simulation.mtm))
    print(path.name, loaded["mtm"].shape)
